In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")

print("train:", train.shape)
print("test:", test.shape)
print("train series:", train_series.shape)
print("test series:", test_series.shape)

In [ ]:
gold = train[TARGETS].notna().all(axis=1)
partial = train[TARGETS].notna().any(axis=1) & ~gold

print("Gold-labelled training studies:", gold.sum())
print("Unlabelled training studies:", (~gold).sum())
print("Partially labelled studies:", partial.sum())

print("\nPositive gold-label counts:")
display(train.loc[gold, TARGETS].sum().sort_values(ascending=False))

print("\nSeries by imaging plane:")
display(train_series["Anatomical_Plane"].value_counts(dropna=False))

print("\nStudies without a listed series:")
print((~train["StudyInstanceUID"].isin(train_series["StudyInstanceUID"])).sum())

In [ ]:
submission = test[["StudyInstanceUID"]].copy()

for target in TARGETS:
    submission[target] = 0.5

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.shape)
display(submission.head())

In [ ]:
import pydicom
import matplotlib.pyplot as plt

example_uid = train.loc[gold, "StudyInstanceUID"].iloc[0]
example_series = train_series[
    train_series["StudyInstanceUID"] == example_uid
].copy()

display(example_series)

series_uid = example_series["SeriesInstanceUID"].iloc[0]
dicom_files = sorted(
    (DATA_DIR / "train_series" / example_uid / series_uid).glob("*.dcm")
)

print("Slices in this series:", len(dicom_files))

middle_file = dicom_files[len(dicom_files) // 2]
dcm = pydicom.dcmread(middle_file)
image = dcm.pixel_array

plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.title(f"Middle slice — {example_series['Anatomical_Plane'].iloc[0]} plane")
plt.show()

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

def load_mri_slice(dicom_path, image_size=224):
    dcm = pydicom.dcmread(dicom_path)
    image = dcm.pixel_array.astype(np.float32)

    low, high = np.percentile(image, [1, 99])
    image = np.clip((image - low) / max(high - low, 1e-6), 0, 1)

    tensor = torch.from_numpy(image)[None, None]  # batch, channel, height, width
    tensor = F.interpolate(
        tensor,
        size=(image_size, image_size),
        mode="bilinear",
        align_corners=False,
    )

    return tensor[0]  # channel, height, width

slice_tensor = load_mri_slice(middle_file)

print("Tensor shape:", tuple(slice_tensor.shape))
print("Value range:", float(slice_tensor.min()), "to", float(slice_tensor.max()))
print("PyTorch GPU available:", torch.cuda.is_available())

In [ ]:
def load_mri_series(dicom_files, num_slices=24, image_size=224):
    if not dicom_files:
        raise ValueError("This series has no DICOM files.")

    indices = np.linspace(0, len(dicom_files) - 1, num_slices).round().astype(int)

    slices = [
        load_mri_slice(dicom_files[index], image_size=image_size)
        for index in indices
    ]

    return torch.stack(slices)  # slices, channel, height, width

series_tensor = load_mri_series(dicom_files)

print("Series tensor shape:", tuple(series_tensor.shape))
print("Selected slices:", len(series_tensor))
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def select_study_series(study_series, max_series=5):
    selected = study_series.copy()

    fluid = selected["Fluid_Sensitive"].fillna(0)
    fat_suppression = selected["Fat_Suppression"].fillna(0)
    selected["_rank"] = 2 * fluid + fat_suppression

    selected = selected.sort_values(
        ["Anatomical_Plane", "_rank"],
        ascending=[True, False],
    )

    # First retain the best acquisition from each plane, then fill remaining slots.
    one_per_plane = selected.groupby("Anatomical_Plane", dropna=False).head(1)
    remaining = selected.drop(one_per_plane.index).sort_values("_rank", ascending=False)

    return pd.concat([one_per_plane, remaining]).head(max_series)


def load_study_tensor(study_uid, series_table, max_series=5, num_slices=24, image_size=224):
    chosen = select_study_series(
        series_table[series_table["StudyInstanceUID"] == study_uid],
        max_series=max_series,
    )

    series_tensors = []
    metadata = []
    series_mask = []

    for _, row in chosen.iterrows():
        files = sorted(
            (
                DATA_DIR
                / "train_series"
                / study_uid
                / row["SeriesInstanceUID"]
            ).glob("*.dcm")
        )

        series_tensors.append(
            load_mri_series(files, num_slices=num_slices, image_size=image_size)
        )

        plane = row["Anatomical_Plane"]
        metadata.append([
            float(plane == "Sagittal"),
            float(plane == "Coronal"),
            float(plane == "Axial"),
            float(row["Fluid_Sensitive"]),
            float(row["Fat_Suppression"]),
        ])
        series_mask.append(1.0)

    while len(series_tensors) < max_series:
        series_tensors.append(
            torch.zeros(num_slices, 1, image_size, image_size)
        )
        metadata.append([0.0] * 5)
        series_mask.append(0.0)

    return (
        torch.stack(series_tensors),      # series, slices, channel, height, width
        torch.tensor(metadata),
        torch.tensor(series_mask),
        chosen,
    )


study_tensor, study_metadata, study_mask, chosen_series = load_study_tensor(
    example_uid,
    train_series,
)

print("Study tensor shape:", tuple(study_tensor.shape))
print("Series mask:", study_mask.tolist())
print("Selected planes:", chosen_series["Anatomical_Plane"].tolist())

In [ ]:
LABEL_PATH = Path(
    "/kaggle/input/datasets/yunusgmsoy/rsna-knee-llm-report-labels/"
    "yunus_llm_labels.csv"
)

llm_labels = pd.read_csv(LABEL_PATH)

print("Label table shape:", llm_labels.shape)
print("Columns:")
print(llm_labels.columns.tolist())

print("\nDuplicate study IDs:", llm_labels["StudyInstanceUID"].duplicated().sum())
print("Studies matched to train.csv:", 
      train["StudyInstanceUID"].isin(llm_labels["StudyInstanceUID"]).sum())

In [ ]:
assert llm_labels["StudyInstanceUID"].duplicated().sum() == 0
assert set(TARGETS).issubset(llm_labels.columns)

pseudo = llm_labels.rename(
    columns={target: f"pseudo_{target}" for target in TARGETS}
)

training = train[["StudyInstanceUID", *TARGETS]].merge(
    pseudo,
    on="StudyInstanceUID",
    how="left",
    validate="one_to_one",
)

for target in TARGETS:
    training[f"target_{target}"] = np.where(
        gold,
        training[target],
        training[f"pseudo_{target}"].fillna(0.5),
    )
    training[f"weight_{target}"] = np.where(gold, 1.0, 0.55)

print("Training studies:", len(training))
print("Expert-labelled studies:", int(gold.sum()))
print("Pseudo-labelled studies:", int((~gold).sum()))
print("Missing pseudo-label cells:", int(
    training[[f"pseudo_{target}" for target in TARGETS]].isna().sum().sum()
))

In [ ]:
from sklearn.model_selection import KFold

training["fold"] = -1

gold_indices = training.index[gold].to_numpy()
splitter = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_number, (_, validation_positions) in enumerate(splitter.split(gold_indices)):
    validation_indices = gold_indices[validation_positions]
    training.loc[validation_indices, "fold"] = fold_number

train_frame = training[training["fold"] != 0].reset_index(drop=True)
valid_frame = training[training["fold"] == 0].reset_index(drop=True)

print("Training studies:", len(train_frame))
print("Validation studies:", len(valid_frame))
print("Validation uses expert labels only:", bool((valid_frame["fold"] == 0).all()))

In [ ]:
def order_dicom_files(dicom_files):
    ordered = []

    for fallback_index, path in enumerate(dicom_files):
        try:
            header = pydicom.dcmread(path, stop_before_pixels=True)
            instance_number = int(getattr(header, "InstanceNumber", fallback_index))
        except Exception:
            instance_number = fallback_index

        ordered.append((instance_number, path))

    return [path for _, path in sorted(ordered)]


def load_mri_series(dicom_files, num_slices=24, image_size=224):
    if not dicom_files:
        return torch.zeros(num_slices, 1, image_size, image_size, dtype=torch.float32)

    dicom_files = order_dicom_files(dicom_files)
    indices = np.linspace(0, len(dicom_files) - 1, num_slices).round().astype(int)

    slices = []

    for index in indices:
        try:
            slices.append(
                load_mri_slice(dicom_files[index], image_size=image_size)
            )
        except Exception:
            # Skip corrupt/unreadable slices while retaining the study.
            continue

    if not slices:
        return torch.zeros(num_slices, 1, image_size, image_size, dtype=torch.float32)

    while len(slices) < num_slices:
        slices.append(slices[-1].clone())

    return torch.stack(slices[:num_slices]).float()

In [ ]:
from torch.utils.data import Dataset, DataLoader

class KneeStudyDataset(Dataset):
    def __init__(self, frame, series_table, max_series=5, num_slices=24, image_size=224):
        self.frame = frame.reset_index(drop=True)
        self.series_table = series_table
        self.max_series = max_series
        self.num_slices = num_slices
        self.image_size = image_size

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        study_uid = row["StudyInstanceUID"]

        images, metadata, series_mask, _ = load_study_tensor(
            study_uid,
            self.series_table,
            max_series=self.max_series,
            num_slices=self.num_slices,
            image_size=self.image_size,
        )

        targets = torch.tensor(
            [row[f"target_{target}"] for target in TARGETS],
            dtype=torch.float32,
        )
        weights = torch.tensor(
            [row[f"weight_{target}"] for target in TARGETS],
            dtype=torch.float32,
        )

        return {
            "images": images,
            "metadata": metadata,
            "series_mask": series_mask,
            "targets": targets,
            "weights": weights,
        }


train_dataset = KneeStudyDataset(train_frame, train_series)
valid_dataset = KneeStudyDataset(valid_frame, train_series)

sample = valid_dataset[0]

print("Training dataset size:", len(train_dataset))
print("Validation dataset size:", len(valid_dataset))
print("One study image shape:", tuple(sample["images"].shape))
print("One study target shape:", tuple(sample["targets"].shape))

In [ ]:
from torch import nn
from torchvision.models import resnet18
from torch.utils.data import DataLoader

RESNET_WEIGHTS = (
    "/kaggle/input/datasets/pytorch/resnet18/resnet18.pth"
)

class KneeMRIModel(nn.Module):
    def __init__(self, num_targets=12):
        super().__init__()

        backbone = resnet18(weights=None)

        checkpoint = torch.load(
            RESNET_WEIGHTS,
            map_location="cpu",
            weights_only=False,
        )
        
        if isinstance(checkpoint, nn.Module):
            state_dict = checkpoint.state_dict()
        else:
            state_dict = checkpoint.get("state_dict", checkpoint)
        
        if any(key.startswith("module.") for key in state_dict):
            state_dict = {
                key.removeprefix("module."): value
                for key, value in state_dict.items()
            }
        
        backbone.load_state_dict(state_dict)
        self.image_encoder = nn.Sequential(*list(backbone.children())[:-1])

        self.metadata_encoder = nn.Sequential(
            nn.Linear(5, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
        )

        self.attention = nn.Sequential(
            nn.Linear(512 + 32, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        self.classifier = nn.Linear(512 + 32, num_targets)

    def forward(self, images, metadata, series_mask):
        images = images.float()
        metadata = metadata.float()
        series_mask = series_mask.float()
        
        batch_size, num_series, num_slices, _, height, width = images.shape

        # Convert grayscale MRI slices to three channels for ResNet.
        x = images.reshape(
            batch_size * num_series * num_slices, 1, height, width
        ).repeat(1, 3, 1, 1)

        image_features = self.image_encoder(x).flatten(1)
        image_features = image_features.reshape(
            batch_size, num_series, num_slices, 512
        ).mean(dim=2)

        metadata_features = self.metadata_encoder(metadata)
        series_features = torch.cat(
            [image_features, metadata_features],
            dim=-1,
        )

        attention_scores = self.attention(series_features).squeeze(-1)
        attention_scores = attention_scores.masked_fill(series_mask == 0, -1e4)
        attention_weights = torch.softmax(attention_scores, dim=1)

        study_features = (
            series_features * attention_weights.unsqueeze(-1)
        ).sum(dim=1)

        return self.classifier(study_features)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KneeMRIModel(num_targets=len(TARGETS)).to(device)

smoke_loader = DataLoader(
    valid_dataset,
    batch_size=1,
    shuffle=False,
)
batch = next(iter(smoke_loader))

model.eval()

with torch.no_grad():
    logits = model(
        batch["images"].to(device),
        batch["metadata"].to(device),
        batch["series_mask"].to(device),
    )

print("Device:", device)
print("Logit shape:", tuple(logits.shape))
print("Target count:", logits.shape[1])

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

SEED = 42
NUM_EPOCHS = 1
CHECKPOINT_EVERY = 200
MAX_TRAIN_STEPS = 20

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / "kneesignal_resnet18_v03_fold0.pt"
RECOVERY_PATH = OUTPUT_DIR / "kneesignal_v03_fold0_recovery.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Rebuild from the pretrained encoder, not the previous smoke-test model.
model = KneeMRIModel(num_targets=len(TARGETS)).to(device)

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    persistent_workers=True,
    prefetch_factor=2,
    generator=generator,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

assert len(train_loader) > 0, "Training dataset is empty."
assert len(valid_loader) > 0, "Validation dataset is empty."

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4,
)


def atomic_save(payload, destination):
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(destination)


def save_recovery(epoch_number, completed_steps):
    atomic_save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch_number,
            "completed_steps": completed_steps,
            "targets": list(TARGETS),
            "seed": SEED,
        },
        RECOVERY_PATH,
    )


for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Training epoch {epoch + 1}",
    )

    for step, batch in enumerate(progress, start=1):
        images = batch["images"].to(device).float()
        metadata = batch["metadata"].to(device).float()
        series_mask = batch["series_mask"].to(device).float()
        targets = batch["targets"].to(device).float()
        weights = batch["weights"].to(device).float()

        optimizer.zero_grad(set_to_none=True)

        logits = model(images, metadata, series_mask)

        loss_per_target = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        # Preserve the intended pseudo-label loss weighting.
        loss = (loss_per_target * weights).mean()

        if not torch.isfinite(loss).item():
            raise RuntimeError(f"Non-finite training loss at step {step}.")

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        progress.set_postfix(loss=f"{running_loss / step:.4f}")

        if step % CHECKPOINT_EVERY == 0:
            save_recovery(epoch + 1, step)
            tqdm.write(f"Recovery checkpoint saved at step {step}.")
            

    # Save completed training BEFORE validation, in case validation fails.
    atomic_save(model.state_dict(), MODEL_PATH)
    save_recovery(epoch + 1, step)
    print(f"Saved trained weights: {MODEL_PATH.name}")

    model.eval()
    valid_predictions = []
    valid_targets = []

    with torch.inference_mode():
        for batch in tqdm(valid_loader, desc="Validating"):
            logits = model(
                batch["images"].to(device).float(),
                batch["metadata"].to(device).float(),
                batch["series_mask"].to(device).float(),
            )

            valid_predictions.append(logits.sigmoid().cpu().numpy())
            valid_targets.append(batch["targets"].cpu().numpy())

    valid_predictions = np.concatenate(valid_predictions)
    valid_targets = np.concatenate(valid_targets)

    assert np.isfinite(valid_predictions).all(), (
        "Validation predictions contain non-finite values."
    )

    aucs = []
    skipped_targets = []

    for target_index, target_name in enumerate(TARGETS):
        y_true = valid_targets[:, target_index]

        if len(np.unique(y_true)) == 2:
            aucs.append(
                roc_auc_score(y_true, valid_predictions[:, target_index])
            )
        else:
            skipped_targets.append(target_name)

    macro_auc = float(np.mean(aucs)) if aucs else float("nan")

    print(
        f"Epoch {epoch + 1} mean weighted training loss: "
        f"{running_loss / step:.4f}"
    )
    print(f"Expert-validation macro AUC: {macro_auc:.4f}")
    print(f"Evaluable targets: {len(aucs)} / {len(TARGETS)}")
    print(f"Targets not evaluable: {skipped_targets}")
    print(f"Saved: {MODEL_PATH.name}")

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

test = pd.read_csv(DATA_DIR / "test.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")

checkpoint_path = Path("/kaggle/working/kneesignal_resnet18_fold0.pt")

print("Test studies:", len(test))
print("Saved model file exists:", checkpoint_path.exists())
print("Trained model still in memory:", "model" in globals())

In [ ]:
from pathlib import Path

print("Model in memory:", "model" in globals())
print("Training dataset in memory:", "train_dataset" in globals())

for name in [
    "kneesignal_resnet18_v03_fold0.pt",
    "kneesignal_v03_fold0_recovery.pt",
]:
    path = Path("/kaggle/working") / name
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")